## GDELT & ACLED -- Structured Event Data Collection

### By:
MGO

### Date:
2026-03-23

### Description:

Collect global event data (GDELT) and conflict/protest data (ACLED) for Colombia.
These two structured event databases let us correlate **online discourse** (from RSS,
Reddit) with **offline events** (protests, political violence, state repression).

- **GDELT**: monitors news media worldwide in 65+ languages with 15-minute updates.
  Provides tone scores that serve as an independent baseline for our pysentimiento results.
- **ACLED**: geocoded conflict and protest events. Critical for DDI Thematic Area 3
  (Early Warning Indicators for Civic Crackdowns).

**APIs:** GDELT DOC API (no auth) + ACLED REST API (requires registration)

**Data source priority:** P1 (Day 2)

**Output:**
- `data/01_raw/gdelt/gdelt_articles_YYYYMMDD.json`
- `data/01_raw/acled/acled_events_YYYYMMDD.json`

## Import libraries

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

# Add src to path for importing project utilities
sys.path.insert(0, str(Path("../../src").resolve()))

from collectors.acled import query_acled
from collectors.gdelt import query_gdelt
from utils.config import load_config

# Load environment variables
load_dotenv(Path("../../.env"))

sns.set_theme(style="darkgrid")

## Configuration

Load filter settings from YAML config files. Each source has its own config
with country filters, event types, and collection parameters.

In [ ]:
# GDELT configuration
gdelt_config = load_config("../../conf/data_collection/gdelt.yml")
gdelt_filters = gdelt_config["filters"]
gdelt_settings = gdelt_config["settings"]

print("GDELT Configuration:")
print(f"  Country: {gdelt_filters['country_name']} ({gdelt_filters['country_code']})")
print(f"  Themes: {gdelt_filters['themes']}")
print(f"  Max records: {gdelt_settings['max_records']}")

# ACLED configuration
acled_config = load_config("../../conf/data_collection/acled.yml")
acled_filters = acled_config["filters"]
acled_settings = acled_config["settings"]

print(f"\nACLED Configuration:")
print(f"  Country: {acled_filters['country']}")
print(f"  Event types: {acled_filters['event_types']}")
print(f"  Lookback: {acled_settings['lookback_days']} days")

# Check ACLED credentials
has_acled_credentials = bool(os.getenv("ACLED_EMAIL") and os.getenv("ACLED_PASSWORD"))
print(f"\nACLED API credentials configured: {has_acled_credentials}")
if not has_acled_credentials:
    print("NOTE: Register at https://acleddata.com for API access (24-48h approval)\n"
          "Set ACLED_EMAIL and ACLED_PASSWORD in .env")

## GDELT Collection

Query the GDELT DOC API for articles about Colombian politics. The API is free
and requires no authentication. It provides:
- Article URLs and titles
- Tone scores (-100 to +100) as an independent sentiment baseline
- Source domains and language metadata

In [ ]:
# Load election keywords for the GDELT query
keywords_config = load_config("../../conf/keywords/election.yml")
election_keywords = keywords_config["keywords"][:10]  # Top 10 for API query
print(f"Using keywords: {election_keywords}")

# Query GDELT (live HTTP request to public API)
try:
    df_gdelt = query_gdelt(
        country=gdelt_filters["country_name"],
        keywords=election_keywords,
        max_records=gdelt_settings["max_records"],
    )
    print(f"Collected {len(df_gdelt)} articles from GDELT")
except Exception as e:
    print(f"GDELT query failed: {e}")
    print("Creating empty DataFrame -- will retry later")
    df_gdelt = pd.DataFrame()

In [ ]:
if not df_gdelt.empty:
    print(f"Shape: {df_gdelt.shape}")
    print(f"Columns: {df_gdelt.columns.tolist()}")
    print(f"\nSample articles:")
    display(df_gdelt[["title", "source_domain", "tone"]].head(10))

    # Tone distribution
    print(f"\nTone statistics:")
    print(df_gdelt["tone"].describe())
    print(f"\nArticles with negative tone: {(df_gdelt['tone'] < 0).sum()}")
    print(f"Articles with positive tone: {(df_gdelt['tone'] > 0).sum()}")
else:
    print("No GDELT data collected -- check API availability")

In [ ]:
# Save GDELT data to raw layer
today = datetime.now().strftime("%Y%m%d")

if not df_gdelt.empty:
    gdelt_output_dir = Path("../../data/01_raw/gdelt")
    gdelt_output_dir.mkdir(parents=True, exist_ok=True)
    gdelt_output_path = gdelt_output_dir / f"gdelt_articles_{today}.json"
    df_gdelt.to_json(gdelt_output_path, orient="records", force_ascii=False, indent=2)
    print(f"Saved {len(df_gdelt)} GDELT articles to {gdelt_output_path}")
else:
    print("No GDELT data to save")

## ACLED Collection

Query the ACLED API for Colombian conflict and protest events. Key event types:
- **Protests** -- peaceful protests, protests with intervention, excessive force
- **Riots** -- mob violence, property destruction
- **Violence against civilians** -- attacks on social leaders, human rights defenders
- **Strategic developments** -- political agreements, policy changes

Each event includes **geocoded coordinates** (latitude, longitude) for geographic analysis.

In [ ]:
# Query ACLED API (requires credentials in .env)
if has_acled_credentials:
    try:
        df_acled = query_acled(
            country=acled_filters["country"],
            event_types=acled_filters["event_types"],
            date_range=("2026-01-01", "2026-03-23"),
            limit=acled_settings["limit"],
        )
        print(f"Collected {len(df_acled)} events from ACLED")
    except Exception as e:
        print(f"ACLED query failed: {e}")
        df_acled = pd.DataFrame()
else:
    # Demonstration with synthetic sample data
    print("Using synthetic sample (set ACLED_EMAIL and ACLED_PASSWORD for live data)")
    df_acled = pd.DataFrame([
        {
            "event_id": "ACL001", "event_date": "2026-03-10",
            "event_type": "Protests", "sub_event_type": "Peaceful protest",
            "actor1": "Students (Colombia)", "actor2": "",
            "location": "Bogota", "latitude": 4.711, "longitude": -74.0721,
            "admin1": "Bogota", "admin2": "", "fatalities": 0,
            "notes": "Protesta estudiantil pacifica en la Plaza de Bolivar",
            "source": "Local media",
        },
        {
            "event_id": "ACL002", "event_date": "2026-03-12",
            "event_type": "Violence against civilians", "sub_event_type": "Attack",
            "actor1": "Unidentified Armed Group", "actor2": "Civilians (Colombia)",
            "location": "Tumaco", "latitude": 1.8065, "longitude": -78.7649,
            "admin1": "Narino", "admin2": "Tumaco", "fatalities": 1,
            "notes": "Asesinato de lider social en zona rural de Tumaco",
            "source": "Indepaz",
        },
        {
            "event_id": "ACL003", "event_date": "2026-03-15",
            "event_type": "Protests", "sub_event_type": "Protest with intervention",
            "actor1": "Labor Groups (Colombia)", "actor2": "Police (Colombia)",
            "location": "Cali", "latitude": 3.4516, "longitude": -76.532,
            "admin1": "Valle del Cauca", "admin2": "Cali", "fatalities": 0,
            "notes": "ESMAD interviene protesta de trabajadores en centro de Cali",
            "source": "El Pais Cali",
        },
        {
            "event_id": "ACL004", "event_date": "2026-03-18",
            "event_type": "Protests", "sub_event_type": "Peaceful protest",
            "actor1": "Indigenous Groups (Colombia)", "actor2": "",
            "location": "Popayan", "latitude": 2.4419, "longitude": -76.6063,
            "admin1": "Cauca", "admin2": "Popayan", "fatalities": 0,
            "notes": "Minga indigena exige cumplimiento de acuerdos con gobierno",
            "source": "CRIC",
        },
        {
            "event_id": "ACL005", "event_date": "2026-03-20",
            "event_type": "Violence against civilians", "sub_event_type": "Attack",
            "actor1": "FARC-EP Dissidents", "actor2": "Civilians (Colombia)",
            "location": "Arauca", "latitude": 7.0847, "longitude": -70.7592,
            "admin1": "Arauca", "admin2": "Arauca", "fatalities": 2,
            "notes": "Ataque contra civiles en zona de conflicto armado",
            "source": "Defensoria del Pueblo",
        },
    ])
    print(f"Sample events: {len(df_acled)}")

In [ ]:
if not df_acled.empty:
    print(f"Shape: {df_acled.shape}")
    print(f"\nEvent type distribution:")
    print(df_acled["event_type"].value_counts().to_string())
    print(f"\nGeographic spread:")
    print(df_acled["admin1"].value_counts().to_string())
    print(f"\nTotal fatalities: {df_acled['fatalities'].sum()}")
    display(df_acled)

In [ ]:
# Save ACLED data to raw layer
if not df_acled.empty:
    acled_output_dir = Path("../../data/01_raw/acled")
    acled_output_dir.mkdir(parents=True, exist_ok=True)
    acled_output_path = acled_output_dir / f"acled_events_{today}.json"
    df_acled.to_json(acled_output_path, orient="records", force_ascii=False, indent=2)
    print(f"Saved {len(df_acled)} ACLED events to {acled_output_path}")
else:
    print("No ACLED data to save")

## Combining Event Data

Merge GDELT and ACLED on date and location where possible to create a unified
event timeline. This lets us see how international media coverage (GDELT) relates
to ground-truth events (ACLED).

In [ ]:
# Build unified event timeline
timeline_events = []

# Add ACLED events to timeline
if not df_acled.empty:
    for _, row in df_acled.iterrows():
        timeline_events.append({
            "date": row["event_date"],
            "source_db": "ACLED",
            "event_type": row["event_type"],
            "location": row.get("location", ""),
            "description": row.get("notes", ""),
            "fatalities": row.get("fatalities", 0),
        })

# Add GDELT articles to timeline (summarized by date)
if not df_gdelt.empty:
    # Parse GDELT dates and group by day
    df_gdelt_copy = df_gdelt.copy()
    df_gdelt_copy["date"] = pd.to_datetime(df_gdelt_copy["published"]).dt.strftime("%Y-%m-%d")
    for date, group in df_gdelt_copy.groupby("date"):
        timeline_events.append({
            "date": date,
            "source_db": "GDELT",
            "event_type": "Media coverage",
            "location": "Colombia (national)",
            "description": f"{len(group)} news articles, avg tone: {group['tone'].mean():.1f}",
            "fatalities": 0,
        })

df_timeline = pd.DataFrame(timeline_events)

if not df_timeline.empty:
    df_timeline = df_timeline.sort_values("date")
    print(f"Unified timeline: {len(df_timeline)} events")
    print(f"\nTimeline preview:")
    for _, event in df_timeline.iterrows():
        marker = "[ACLED]" if event["source_db"] == "ACLED" else "[GDELT]"
        print(f"  {event['date']} {marker} {event['location']}: {event['description'][:80]}")
else:
    print("No events to combine -- both sources returned empty")

## Collection Summary

In [ ]:
print("=" * 60)
print("GDELT & ACLED COLLECTION SUMMARY")
print("=" * 60)

# Row counts per source
gdelt_count = len(df_gdelt) if not df_gdelt.empty else 0
acled_count = len(df_acled) if not df_acled.empty else 0
print(f"\nGDELT articles: {gdelt_count}")
print(f"ACLED events:   {acled_count}")
print(f"Total records:  {gdelt_count + acled_count}")

# Date range coverage
if not df_gdelt.empty:
    print(f"\nGDELT date range: {df_gdelt['published'].min()} to {df_gdelt['published'].max()}")
if not df_acled.empty:
    print(f"ACLED date range: {df_acled['event_date'].min()} to {df_acled['event_date'].max()}")

# Event type breakdown
if not df_acled.empty:
    print(f"\nACLED event type breakdown:")
    print(df_acled["event_type"].value_counts().to_string())

# Geographic distribution
if not df_acled.empty and "admin1" in df_acled.columns:
    print(f"\nGeographic distribution (ACLED events by department):")
    print(df_acled["admin1"].value_counts().to_string())

In [ ]:
# Visualize event type breakdown
if not df_acled.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Event type bar chart
    df_acled["event_type"].value_counts().plot(
        kind="bar", ax=axes[0], color="steelblue"
    )
    axes[0].set_title("ACLED Events by Type")
    axes[0].set_xlabel("Event Type")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis="x", rotation=45)

    # Geographic distribution
    df_acled["admin1"].value_counts().plot(
        kind="barh", ax=axes[1], color="coral"
    )
    axes[1].set_title("ACLED Events by Department")
    axes[1].set_xlabel("Count")

    plt.tight_layout()
    plt.show()

## Notes & Next Steps

**Data quality observations:**
- GDELT tone scores provide an independent sentiment baseline to compare with pysentimiento
- ACLED geocoded coordinates enable geographic overlay analysis
- Some GDELT articles may overlap with our RSS collection -- deduplication needed

**Coverage gaps:**
- GDELT coverage depends on online media; rural events may be underreported
- ACLED has a reporting lag of days to weeks for some event types
- Neither source captures social media discourse -- that comes from our RSS/Reddit collectors

**How this contextualizes the social media data:**
- When we detect a spike in negative sentiment online, ACLED can tell us whether
  a real-world event (protest, violence) triggered it
- Conversely, online discourse spikes BEFORE ACLED events may indicate early warning signals
- GDELT tone trends over time provide a media-level baseline for sentiment comparison

**Next steps:**
1. Deduplicate GDELT articles that overlap with RSS collection
2. Use ACLED geographic data in the analysis summary (notebook 4-output/02)
3. Build temporal correlation between online sentiment and offline events

## References

- GDELT Project: https://gdeltproject.org
- GDELT DOC API: https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/
- ACLED: https://acleddata.com
- ACLED Colombia profile: https://acleddata.com/dashboard/#/dashboard/CO
- CIVICUS Monitor Colombia: https://monitor.civicus.org/country/colombia/